# `tensorium` tutorial, part 5: tensor operators

This notebook develops the operator point of view of the library. Covariant derivatives, multiplication maps, contractions, tensor products and internal matrix actions can all be represented as `TensorOperator` objects and combined before being applied to fields.


In [1]:
from sympy import simplify, symbols
from tensorium import *

S2 = Manifold("S^2", 2)
U_N = OpenSet("U_N", S2)  # S^2 without the north pole
U_S = OpenSet("U_S", S2)  # S^2 without the south pole
x, y = symbols("x y", real=True)
u, v = symbols("u v", real=True)
X_N = Chart("X_N", U_N, (x, y))
X_S = Chart("X_S", U_S, (u, v), relations={X_N: (u/(u**2 + v**2), v/(u**2 + v**2))})
atlas = Atlas(S2, [X_N, X_S])
S2.set_atlas(atlas)
xN, yN = X_N.symbols
uS, vS = X_S.symbols
f_N_local = LocalTensorField(X_N, (0, 0), 1/(1 + xN**2 + yN**2))
f_S_local = LocalTensorField(X_S, (0, 0), (uS**2 + vS**2)/(1 + uS**2 + vS**2))
f = TensorField(S2, (0, 0), {X_N: f_N_local, X_S: f_S_local}, ())
V_N_local = LocalTensorField(X_N, (1, 0), [yN, -xN])
V_S_local = LocalTensorField(X_S, (1, 0), [vS, -uS])
V = TensorField(S2, (1, 0), {X_N: V_N_local, X_S: V_S_local}, (1,))
omega_N_local = LocalTensorField(X_N, (0, 1), [xN, yN])
omega_S_local = LocalTensorField(X_S, (0, 1), [uS, vS])
omega = OneForm(S2, {X_N: omega_N_local, X_S: omega_S_local})
lambda_N = 4/(1 + xN**2 + yN**2)**2
lambda_S = 4/(1 + uS**2 + vS**2)**2
g_N_local = LocalCovariantMetricTensor(X_N, (lambda_N, 0, lambda_N))
g_S_local = LocalCovariantMetricTensor(X_S, (lambda_S, 0, lambda_S))
g = CovariantMetricTensor(S2, {X_N: g_N_local, X_S: g_S_local})
g_inv = g.inverse()
S2_metric = MetricManifold(S2, covariant_metric=g, contravariant_metric=g_inv)
f_metric = TensorField(S2_metric, (0, 0), f.local_representations, ())
V_metric = TensorField(S2_metric, (1, 0), V.local_representations, (1,))
omega_metric = OneForm(S2_metric, omega.local_representations)
Gamma = LeviCivitaConnection(S2_metric)
alpha_N_local = LocalTensorField(X_N, (0, 1), [xN, yN])
alpha_S_local = LocalTensorField(X_S, (0, 1), [uS, vS])
alpha = OneForm(S2_metric, {X_N: alpha_N_local, X_S: alpha_S_local})
zero_oneform = 0*alpha
A_gauge = GaugeConnection.from_components(S2_metric, [[zero_oneform, alpha], [-alpha, zero_oneform]])
one_metric = TensorField(S2_metric, (0, 0), {
    X_N: LocalTensorField(X_N, (0, 0), 1),
    X_S: LocalTensorField(X_S, (0, 0), 1),
}, ())
xi = TensorMultiplet([one_metric, f_metric], internal_variance=1)
D_A = CovariantDerivative(gauge_connection=A_gauge)
D_full = CovariantDerivative(affine_connection=Gamma, gauge_connection=A_gauge)


## 5.1. Tensor operators as algebraic objects

Up to this point we have used covariant derivatives as functions that are immediately applied to fields. The library also lets us keep them as `TensorOperator` objects. This is useful because operators can be combined before acting on a field.

A `TensorOperator` stores four pieces of information: its free geometric indices, its arity, the Python action that is executed when the operator is applied, and an optional input/output signature. For example, a covariant derivative has one free covariant index,

$$
D_\mu:\mathcal{T}^{r}_{s}(S^2)\otimes E^{\otimes a}\otimes(E^*)^{\otimes b}
\longrightarrow
\mathcal{T}^{r}_{s+1}(S^2)\otimes E^{\otimes a}\otimes(E^*)^{\otimes b}.
$$

The following cells introduce the basic operator constructors and then combine them into more structured objects.

### 5.2. Basic operators

The most elementary operators are the identity, multiplication by a fixed field, tensor product and contraction. The tensor-product operator is binary, while the others shown here are unary.

In [2]:
Id = TensorOperator.identity_operator(name="Id")
M_f = TensorOperator.multiplication_operator(f_metric, name="M_f")
TP = TensorOperator.tensor_product_operator(name=r"\otimes")
Trace_01 = TensorOperator.contraction_operator(0, 1, name=r"\operatorname{Tr}_{01}")

Display(Id, name="Id")
Display(M_f, name=r"M_f")
Display(TP, name=r"\otimes")
Display(Trace_01, name=r"\operatorname{Tr}_{01}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<!-- small-note-html -->
<div style="font-size:0.92em; line-height:1.35; padding:0.45em 0.70em; margin:0.45em 0 0.70em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
  <p style="margin:0.18em 0;">The operator <code>M_f</code> sends a field <code>T</code> to <code>T*f</code>. The tensor-product operator sends <code>(A,B)</code> to <code>A⊗B</code>. The contraction operator applies the usual tensor contraction to two geometric indices of its argument.</p>
</div>

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">

When an operator is applied to a concrete field, the field signature is compared with the operator input signature.

</div>

In [3]:
Mf_xi = M_f(xi)
xi_tensor_omega = TP(xi, omega_metric)
trace_example = Trace_01(V_metric.tensor_product(omega_metric))

Display(Mf_xi, chart=X_N, name=r"M_f\xi")
Display(xi_tensor_omega, chart=X_N, name=r"\xi\otimes\omega")
Display(trace_example, chart=X_N, name=r"\operatorname{Tr}(V\otimes\omega)")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5.3. Algebra of unary operators

Unary operators with compatible free indices can be added, subtracted and multiplied by scalar factors. They can also be composed. If `A.compose(B)` is formed, then `B` acts first:
$
(A\circ B)(T)=A(B(T)).
$

For covariant derivatives this produces higher-order differential operators with several free derivative indices.

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">

 Composition checks that the formal output of the inner operator is compatible with the input of the outer one. Addition requires the same arity, the same free-index variance and compatible output signatures. If some information is symbolic or unknown, the library allows the construction and lets the actual evaluation decide.

</div>

In [4]:
D = D_full
D2 = D.compose(D)
D_plus_M = D + M_f.compose(D)

Display(D, name="D")
Display(D2, name=r"D\circ D")
Display(D_plus_M, name=r"D+M_f\circ D")

D2_xi = D2(xi)
Display(D2_xi, chart=X_N, name=r"D^2\xi")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5.4. Raising, lowering and contracting operator indices

Operator indices can be manipulated before the operator is applied. Since the derivative index of $D_\mu$ is covariant, the metric can raise it to form $D^\mu$. Contracting $D^\mu$ with $D_\mu$ gives a Laplace-type operator,

$$
D^\mu D_\mu.
$$

In [5]:
D_up = D.raise_index()
gauge_laplacian = D_up.contract_with(D)
lap_xi = gauge_laplacian(xi)

Display(D_up, name=r"D^\mu")
Display(gauge_laplacian, name=r"D^\mu D_\mu")
Display(lap_xi, chart=X_N, name=r"D^\mu D_\mu\xi")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5.5. Commutators

The commutator of two unary operators is implemented as

$$
[A,B]=A\circ B-B\circ A.
$$

For instance, if $M_f$ denotes multiplication by a scalar field, then $[D,M_f]$ measures the failure of the covariant derivative to commute with multiplication by $f$.

In [6]:
D_comm_Mf = D.commutator(M_f, name=r"[D,M_f]")
comm_xi = D_comm_Mf(xi)

Display(D_comm_Mf, name=r"[D,M_f]")
Display(comm_xi, chart=X_N, name=r"[D,M_f]\xi")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5.6. Internal actions as operators

Matrix-valued tensor fields can also be converted into operators. If

$$
A=A^{[A]}{}_{[B]\mu}
$$

is a matrix-valued one-form, `internal_action_operator(A)` builds the unary operator

$$
(A\xi)^{[A]}{}_{\mu}=\sum_B A^{[A]}{}_{[B]\mu}\xi^{[B]}.
$$

The free geometric indices of the matrix-valued field become free indices of the operator.

In [7]:
A_operator = internal_action_operator(A_gauge.internal_tensor_field, name="A")
A_xi = A_operator(xi)

Display(A_operator, name="A")
Display(A_xi, chart=X_N, name=r"A\xi")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5.7. A Dirac-type operator

As a final example, we build a schematic Dirac-type operator. We introduce a vector of matrix-valued fields $\gamma^\mu$ and use `contract_with` to contract its free index with the derivative index:

$$
\mathcal{D}\xi = \gamma^\mu D_\mu\xi.
$$

This is not intended to be a complete spin-geometry implementation, but it illustrates why it is useful to treat covariant derivatives and internal matrix actions as composable operators.

In [8]:
# A simple vector-valued internal matrix. It plays the role of gamma^mu.
e_x_N = LocalTensorField(X_N, (1, 0), [1, 0])
e_y_N = LocalTensorField(X_N, (1, 0), [0, 1])
e_u_S = LocalTensorField(X_S, (1, 0), [1, 0])
e_v_S = LocalTensorField(X_S, (1, 0), [0, 1])

gamma_x = TensorField(S2_metric, (1, 0), {X_N: e_x_N, X_S: e_u_S}, (1,))
gamma_y = TensorField(S2_metric, (1, 0), {X_N: e_y_N, X_S: e_v_S}, (1,))
zero_vector = 0*gamma_x

gamma = ValuedTensorField([[zero_vector, gamma_x],
                             [gamma_y, zero_vector]],
                            internal_shape=(2, 2), internal_variance=(1, -1))

Gamma_operator = internal_action_operator(gamma, name=r"\gamma")
Dirac = Gamma_operator.contract_with(D)
Dirac_xi = Dirac(xi)

Display(gamma, chart=X_N, name=r"\gamma")
Display(Gamma_operator, name=r"\gamma^\mu")
Display(Dirac, name=r"\mathcal{D}")
Display(Dirac_xi, chart=X_N, name=r"\mathcal{D}\xi")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>